# AI-Based Handwritten Digit Recognition System
### Using Convolutional Neural Networks (CNN) on MNIST Dataset
---

## 1. Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.metrics import confusion_matrix, classification_report

print('TensorFlow version:', tf.__version__)
print('GPU Available:', tf.config.list_physical_devices('GPU'))

## 2. Load and Explore MNIST Dataset

In [ ]:
# Load dataset
(X_train_raw, y_train), (X_test_raw, y_test) = tf.keras.datasets.mnist.load_data()

print(f'Training set   : {X_train_raw.shape}  labels: {y_train.shape}')
print(f'Test set       : {X_test_raw.shape}  labels: {y_test.shape}')
print(f'Pixel range    : {X_train_raw.min()} – {X_train_raw.max()}')
print(f'Classes        : {np.unique(y_train)}')

In [ ]:
# Visualise sample images
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for digit in range(10):
    idx = np.where(y_train == digit)[0][0]
    axes[0, digit].imshow(X_train_raw[idx], cmap='gray')
    axes[0, digit].set_title(str(digit))
    axes[0, digit].axis('off')
    idx2 = np.where(y_train == digit)[0][1]
    axes[1, digit].imshow(X_train_raw[idx2], cmap='gray')
    axes[1, digit].axis('off')
plt.suptitle('Sample MNIST Images (two per class)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Class distribution
unique, counts = np.unique(y_train, return_counts=True)
plt.figure(figsize=(8, 4))
plt.bar(unique, counts, color='steelblue', edgecolor='black')
plt.title('Class Distribution (Training Set)')
plt.xlabel('Digit'); plt.ylabel('Count')
plt.xticks(range(10))
plt.show()

## 3. Preprocess Data

In [ ]:
# Normalize: 0-255 → 0-1
X_train = X_train_raw.astype('float32') / 255.0
X_test  = X_test_raw.astype('float32')  / 255.0

# Reshape for CNN input: (samples, 28, 28, 1)
X_train = X_train.reshape(-1, 28, 28, 1)
X_test  = X_test.reshape(-1, 28, 28, 1)

# One-hot encode labels
y_train_cat = to_categorical(y_train, 10)
y_test_cat  = to_categorical(y_test,  10)

print('X_train shape:', X_train.shape)
print('y_train_cat shape:', y_train_cat.shape)
print('Sample label:', y_train[0], '→ one-hot:', y_train_cat[0])

## 4. Build CNN Model

In [ ]:
def build_cnn():
    model = models.Sequential([
        # Convolutional Block 1
        layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(28,28,1)),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.25),

        # Convolutional Block 2
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.25),

        # Fully Connected
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ], name='CNN_Digit_Recognizer')

    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

cnn_model = build_cnn()
cnn_model.summary()

## 5. Train the Model

In [ ]:
callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_cnn_model.keras', save_best_only=True, verbose=1)
]

history = cnn_model.fit(
    X_train, y_train_cat,
    epochs=15,
    batch_size=128,
    validation_data=(X_test, y_test_cat),
    callbacks=callbacks,
    verbose=1
)

## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     label='Train',      color='steelblue')
axes[0].plot(history.history['val_accuracy'], label='Validation', color='orange')
axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train',      color='steelblue')
axes[1].plot(history.history['val_loss'], label='Validation', color='orange')
axes[1].set_title('Loss'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Evaluate on Test Set

In [ ]:
test_loss, test_acc = cnn_model.evaluate(X_test, y_test_cat, verbose=0)
print(f'Test Accuracy : {test_acc*100:.2f}%')
print(f'Test Loss     : {test_loss:.4f}')

In [ ]:
# Predictions
y_pred = np.argmax(cnn_model.predict(X_test), axis=1)

# Classification report
print(classification_report(y_test, y_pred, target_names=[str(i) for i in range(10)]))

## 8. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.title('Confusion Matrix', fontsize=16)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout()
plt.show()

## 9. Sample Predictions

In [ ]:
indices = np.random.choice(len(X_test), 25, replace=False)
preds   = np.argmax(cnn_model.predict(X_test[indices]), axis=1)

fig, axes = plt.subplots(5, 5, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test[indices[i]].reshape(28,28), cmap='gray')
    color = 'green' if preds[i] == y_test[indices[i]] else 'red'
    ax.set_title(f'P:{preds[i]}  T:{y_test[indices[i]]}', color=color, fontsize=8)
    ax.axis('off')
plt.suptitle('Predictions  (Green=Correct, Red=Wrong)', y=1.01)
plt.tight_layout()
plt.show()

## 10. Misclassified Examples

In [ ]:
wrong = np.where(y_pred != y_test)[0]
print(f'Total errors: {len(wrong)} / {len(y_test)}  ({len(wrong)/len(y_test)*100:.2f}%)')

fig, axes = plt.subplots(4, 5, figsize=(10, 8))
for i, ax in enumerate(axes.flat):
    if i >= 20: ax.axis('off'); continue
    idx = wrong[i]
    ax.imshow(X_test[idx].reshape(28,28), cmap='gray')
    ax.set_title(f'P:{y_pred[idx]}  T:{y_test[idx]}', color='red', fontsize=8)
    ax.axis('off')
plt.suptitle('Misclassified Examples')
plt.tight_layout()
plt.show()

## 11. MLP Comparison

In [ ]:
mlp = models.Sequential([
    layers.Flatten(input_shape=(28,28,1)),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])
mlp.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
mlp.fit(X_train, y_train_cat, epochs=10, batch_size=128,
        validation_data=(X_test, y_test_cat), verbose=0)

_, cnn_acc = cnn_model.evaluate(X_test, y_test_cat, verbose=0)
_, mlp_acc = mlp.evaluate(X_test, y_test_cat, verbose=0)

print(f'CNN Accuracy : {cnn_acc*100:.2f}%')
print(f'MLP Accuracy : {mlp_acc*100:.2f}%')

plt.bar(['CNN', 'MLP'], [cnn_acc*100, mlp_acc*100], color=['steelblue','coral'])
plt.title('CNN vs MLP Accuracy')
plt.ylabel('Test Accuracy (%)')
plt.ylim(95, 100)
plt.show()

## 12. Predict Custom Image

In [ ]:
from PIL import Image, ImageOps

def predict_image(model, path):
    img = Image.open(path).convert('L')
    img = ImageOps.invert(img)
    img = img.resize((28, 28))
    arr = np.array(img).astype('float32') / 255.0
    arr = arr.reshape(1, 28, 28, 1)
    probs  = model.predict(arr)[0]
    digit  = np.argmax(probs)
    conf   = probs[digit] * 100
    plt.imshow(arr.reshape(28,28), cmap='gray')
    plt.title(f'Predicted: {digit}  Confidence: {conf:.1f}%')
    plt.axis('off'); plt.show()
    return digit, conf

# Uncomment and set your image path:
# predict_image(cnn_model, 'my_digit.png')